# Data Merging — Stage 4 Assembly 06: Floor Diagnostic

## Input
- All seven pre-normalisation source tables (the same set fed to Stage 3 notebooks 04/05): `Data/Data_Collection/Final/Stage_3_Cleaning/{agg_market_daily_means, agg_market_daily_full_moments, weekly_raw, agg_market_monthly_means, agg_market_monthly_full_moments}.parquet` and `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/{panel_stock_daily_engineered, panel_stock_monthly_engineered}.parquet`
- `lib.normalise.robust_scale_ladder`, `lib.review`
- (Optional, cross-referenced at the end) `Data/Data_Collection/Final/Stage_4_Normalised/decisions_aggregate.csv` and `Data/Data_Collection/Final/Stage_4_Normalised_Panel/decisions_panel.csv`

## Purpose
A **standalone, read-only diagnostic notebook** that answers one narrow but important question: in the expanding z-score computation used throughout Stage 3/4 (`robust_expanding_zscore` in `lib/normalise.py`), the denominator is `max(running_sd, 0.1 × sigma_f, 1e-8)` — a floor mechanism meant to prevent division by a collapsed or near-zero running standard deviation. This notebook checks, for every feature in every table, **how often that floor actually activated** (i.e., won the `max()` instead of the genuine running standard deviation), rather than assuming the floor is harmless boilerplate.

As stated in the header comment: this notebook reads nothing the main pipeline writes, and writes nothing the main pipeline reads — it's purely an audit tool that can be dropped in or removed without affecting the actual data flow. It deliberately **replicates the normalisation logic from `lib/normalise.py` exactly** (same warm-up period, same robust-scale "rung" ladder, same ±10 contribution cap, same Welford online-variance merge) rather than importing and reusing the production z-score function directly — the one addition here is instrumentation: recording, per feature per date, which of the three candidate terms (running sd, floor, epsilon) actually won the `max()`.

## Method

### `date_slices()`
A small utility assuming rows are already sorted by date: returns the `(start, end)` row-index boundaries for each distinct date, used to process the data date-by-date without repeatedly filtering the full array.

### `diagnose()` — the core per-table routine
For a single table, this function:

1. **Determines each feature's warm-up end.** Walks forward date-by-date, counting how many *dates* each feature has any finite observation on, and records the date index at which each feature first reaches `min_dates` — exactly mirroring the production warm-up logic.
2. **Computes the warm-up-period robust scale (`sigma_f`) and rung** via `robust_scale_ladder`, on the block of data up to each feature's own warm-up end — again matching production exactly.
3. **Seeds the running Welford state** (`w_n`, `w_mean`, `w_M2`) from the warm-up block, after clipping values to `median ± CAP × sigma_f` (the same ±10-sigma-equivalent cap used in production).
4. **Walks forward date-by-date after warm-up**, and at each step:
   - Computes the current running standard deviation from the Welford state.
   - Determines which of `floor` (`0.1 × sigma_f`) or the running `sd` is larger, recording a `floor_wins` flag whenever the floor exceeds both the running sd and epsilon.
   - Separately tracks `eps_wins` — cases where even the epsilon floor (`1e-8`) exceeds the running sd, a more extreme degeneracy than the ordinary floor case.
   - Tracks `min_ratio` — the minimum value ever observed of `running_sd / sigma_f`, which quantifies *how close* the running sd came to needing the floor, even on dates where it didn't actually need it. This is useful because a feature that never triggered the floor but consistently runs at, say, 15% of `sigma_f` is one narrow move away from doing so.
   - Updates the running Welford state using the same clip-then-merge logic as production, so the diagnostic's own running statistics track the production pipeline's exactly.

Returns one row per feature (restricted to features that actually reached warm-up) with: which rung its `sigma_f` came from, the floor value itself, count of active (post-warm-up) dates, count of floor-bound dates, count of epsilon-bound dates, percentage of dates floor-bound, and the minimum sd/sigma_f ratio ever observed.

## Reporting

- **Per-table summary line** — for each of the seven tables, prints feature count, how many features ever had the floor bind on at least one date, how many ever hit the epsilon floor, and the single smallest sd/sigma_f ratio observed across all its features.
- **Global summary** — total features examined, count and percentage where the floor *ever* bound, count where epsilon ever bound, total floor-bound feature-dates and total active feature-dates across the whole pipeline.
- **Distribution of `min(running_sd / sigma_f)`** — full percentile breakdown (p0 through p100), giving a sense of how close to the floor the *typical* feature ever got, not just the worst cases.
- **"How close" threshold table** — for several ratio thresholds (0.05 through 1.0), counts how many features ever got within that factor of the floor — a graduated view of near-miss severity, not just a binary "did it bind" answer.
- **Worst 40 floor-bound features**, ranked by percentage of active dates on which the floor bound, with rung, sigma_f, and ratio details for each.
- **Explicit null-result framing:** if the floor *never* bound for any feature on any date, the notebook says so directly and states the practical consequence plainly — "It is inert in this pipeline: removing it would not change one z-score." This is a deliberately falsifiable, stated conclusion rather than a silently-passed check.

## Cross-Reference Against Drop Decisions
If any features *did* trigger the floor, and the Stage 4 decision files (`decisions_aggregate.csv`, `decisions_panel.csv`) are available, the notebook merges the floor-bound feature list against those decisions to answer: **were the floor-bound features already caught and dropped by the ordinary statistical rules anyway?** Reports a value-count breakdown of `action` (kept/dropped) among floor-bound features, plus the full detail table (floor-bound percentage, standardized-deviation-excluding-capped-values, action, and which rule fired) for manual inspection. This closes the loop on whether the floor mechanism is catching anything the rest of the pipeline wasn't already going to catch on its own.

## Output
- `Data/Data_Collection/Final/Stage_4_Normalised/floor_diagnostic.csv` — one row per feature across all seven tables, with rung, sigma_f, floor value, active/floor-bound/eps-bound date counts, percentage floor-bound, and minimum observed sd/sigma_f ratio.

No other pipeline files are read or written; all summary output is printed for inspection only, aside from the single CSV above.

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FLOOR DIAGNOSTIC  --  standalone, read-only, writes one CSV
#
# Answers: for each feature, on how many dates did the denominator floor
#          (0.1 * sigma_f) exceed the running expanding standard deviation,
#          and therefore get used INSTEAD of it?
#
# Replicates lib/normalise.py exactly: same warm-up, same rung ladder, same
# +-10 contribution cap, same Welford merge. The only addition is that it
# records, per feature per date, which of the three terms won the max().
#
# Put this in a NEW notebook:  Code/Data_Merging/Stage_3_Cleaning/06_floor_diagnostic.ipynb
# It reads nothing that the pipeline writes and writes nothing the pipeline reads.
# ══════════════════════════════════════════════════════════════════════════════

import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.normalise import robust_scale_ladder
import lib.review as rv

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 300)

# ── PATHS -- CHECK THESE MATCH YOUR 04 / 05 NOTEBOOKS ────────────────────────
ROOT     = Path('../../../Data/Data_Collection/Final')
AGG_IN   = ROOT / 'Stage_3_Cleaning'                                  # 04 reads here
PANEL_IN = ROOT / 'Stage_1_5_Validation_and_Feature_Engineering'      # 05 reads here
OUT_DIR  = ROOT / 'Stage_4_Normalised'

CAP   = 10.0
FLOOR_MULT = 0.1
EPS   = 1e-8

TABLES = [
    # (filename,                             in_dir,   min_dates, group_col, meta)
    ('agg_market_daily_means',               AGG_IN,   252, None,     ['date', 'target_daily_return']),
    ('agg_market_daily_full_moments',        AGG_IN,   252, None,     ['date', 'target_daily_return']),
    ('weekly_raw',                           AGG_IN,    52, None,     ['date']),
    ('agg_market_monthly_means',             AGG_IN,    24, None,     ['date', 'target_monthly_return']),
    ('agg_market_monthly_full_moments',      AGG_IN,    24, None,     ['date', 'target_monthly_return']),
    ('panel_stock_daily_engineered',         PANEL_IN, 252, 'permno', ['permno', 'date', 'dlyret', 'dlycap']),
    ('panel_stock_monthly_engineered',       PANEL_IN,  24, 'permno', ['permno', 'date', 'month_end_cap']),
]

BINARIES = ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
            'curve_inverted_3m10y', 'credit_stress']


def date_slices(dates):
    """[(start, end)] per distinct date, assuming rows are sorted by date."""
    d = dates.to_numpy()
    change = np.flatnonzero(d[1:] != d[:-1]) + 1
    bounds = np.concatenate([[0], change, [len(d)]])
    return list(zip(bounds[:-1], bounds[1:]))


def diagnose(tag, in_dir, min_dates, meta):
    path = in_dir / f'{tag}.parquet'
    if not path.exists():
        print(f'  !! {tag}: not found at {path}')
        return None

    df = pd.read_parquet(path).sort_values('date', kind='stable').reset_index(drop=True)
    feats = [c for c in df.columns if c not in meta and c not in BINARIES]
    raw = df[feats].to_numpy(dtype=np.float64)
    slices = date_slices(df['date'])
    n_dates, n_feat = len(slices), len(feats)

    # ── which date index does each feature reach its warm-up on? ─────────────
    has = np.isfinite(raw)
    dates_with_data = np.zeros(n_feat, dtype=np.int64)
    warm_end = np.full(n_feat, -1, dtype=np.int64)
    for d, (s, e) in enumerate(slices):
        got = has[s:e].any(axis=0)
        dates_with_data += got
        newly = (warm_end < 0) & (dates_with_data >= min_dates)
        warm_end[newly] = d
    reached = warm_end >= 0

    # ── warm-up: median, ladder, trim, seed ──────────────────────────────────
    sigma_f = np.full(n_feat, np.nan)
    rung    = np.zeros(n_feat, dtype=np.int8)
    w_n     = np.zeros(n_feat)
    w_mean  = np.zeros(n_feat)
    w_M2    = np.zeros(n_feat)

    for j in range(n_feat):
        if not reached[j]:
            continue
        end = slices[warm_end[j]][1]
        block = raw[:end, j]
        v = block[np.isfinite(block)]
        if len(v) < 2:
            reached[j] = False
            continue
        med = float(np.median(v))
        sf, rg = robust_scale_ladder(v, EPS)
        sigma_f[j], rung[j] = sf, rg
        clean = np.clip(block, med - CAP * sf, med + CAP * sf)
        clean = np.where(np.isfinite(block), clean, np.nan)
        cv = clean[np.isfinite(clean)]
        w_n[j]    = len(cv)
        w_mean[j] = float(cv.mean())
        w_M2[j]   = float(cv.var(ddof=1) * (len(cv) - 1)) if len(cv) > 1 else 0.0

    floor = FLOOR_MULT * np.where(np.isfinite(sigma_f), sigma_f, 0.0)

    # ── running: record which term won the max() on every date ───────────────
    n_floor  = np.zeros(n_feat, dtype=np.int64)   # floor beat the running sd
    n_eps    = np.zeros(n_feat, dtype=np.int64)   # eps beat both
    n_active = np.zeros(n_feat, dtype=np.int64)
    min_ratio = np.full(n_feat, np.inf)           # min(running_sd / sigma_f)

    for d, (s, e) in enumerate(slices):
        x = raw[s:e]
        nan = ~np.isfinite(x)
        active = (d > warm_end) & reached
        if not active.any():
            continue

        with np.errstate(invalid='ignore', divide='ignore'):
            sd = np.sqrt(np.where(w_n > 1, w_M2 / np.maximum(w_n - 1, 1), 0.0))

        floor_wins = active & (floor > sd) & (floor > EPS)
        eps_wins   = active & (EPS > sd) & (EPS >= floor)
        n_floor  += floor_wins
        n_eps    += eps_wins
        n_active += active

        with np.errstate(invalid='ignore', divide='ignore'):
            r = np.where(active & (sigma_f > 0), sd / sigma_f, np.inf)
        min_ratio = np.minimum(min_ratio, r)

        denom = np.maximum(np.maximum(sd, floor), EPS)
        upd = (~nan) & active
        batch_n = upd.sum(axis=0).astype(np.float64)
        live = batch_n > 0
        x_cap = np.clip(x, w_mean - CAP * denom, w_mean + CAP * denom)

        bsum  = np.nansum(np.where(upd, x_cap, np.nan), axis=0)
        bmean = np.where(live, bsum / np.maximum(batch_n, 1.0), 0.0)
        bdev  = np.where(upd, x_cap - bmean, 0.0)
        bM2   = np.nansum(bdev ** 2, axis=0)

        comb  = w_n + batch_n
        safe  = np.where(comb > 0, comb, 1.0)
        delta = np.where(live, bmean - w_mean, 0.0)
        w_mean = np.where(live, w_mean + delta * batch_n / safe, w_mean)
        w_M2   = np.where(live, w_M2 + bM2 + delta ** 2 * w_n * batch_n / safe, w_M2)
        w_n    = np.where(live, comb, w_n)

    out = pd.DataFrame({
        'table_source': tag,
        'feature':      feats,
        'rung':         rung,
        'sigma_f':      sigma_f,
        'floor':        floor,
        'n_active_dates': n_active,
        'n_floor_bound':  n_floor,
        'n_eps_bound':    n_eps,
        'pct_floor_bound': np.where(n_active > 0, n_floor / np.maximum(n_active, 1), np.nan),
        'min_sd_over_sigma_f': np.where(np.isfinite(min_ratio), min_ratio, np.nan),
    })
    return out[reached].reset_index(drop=True)


# ══════════════════════════════════════════════════════════════════════════════
print('=' * 96)
print('FLOOR DIAGNOSTIC   denom = max(running_sd, 0.1 * sigma_f, 1e-8)')
print('=' * 96)
print('  n_floor_bound = dates where 0.1*sigma_f was used INSTEAD of the running sd')
print('  min_sd_over_sigma_f = how close the running sd ever got to the floor')
print(f'  (the floor binds when this ratio drops below {FLOOR_MULT})\n')

parts = []
for tag, in_dir, md, gcol, meta in TABLES:
    r = diagnose(tag, in_dir, md, meta)
    if r is None:
        continue
    parts.append(r)
    nb = int((r['n_floor_bound'] > 0).sum())
    ne = int((r['n_eps_bound'] > 0).sum())
    print(f'  {tag:<34} {len(r):>5} features   '
          f'floor bound on {nb:>4}   eps bound on {ne:>3}   '
          f'min ratio {r["min_sd_over_sigma_f"].min():.4f}')

res = pd.concat(parts, ignore_index=True)
res.to_csv(OUT_DIR / 'floor_diagnostic.csv', index=False)

print('\n' + '=' * 96)
print('SUMMARY')
print('=' * 96)
tot = len(res)
ever = int((res['n_floor_bound'] > 0).sum())
print(f'  features examined                        : {tot:,}')
print(f'  features where the floor EVER bound      : {ever:,}  ({ever/tot:.2%})')
print(f'  features where eps ever bound            : {int((res["n_eps_bound"]>0).sum()):,}')
print(f'  total floor-bound feature-dates          : {int(res["n_floor_bound"].sum()):,}')
print(f'  total active feature-dates               : {int(res["n_active_dates"].sum()):,}')

print(f'\n  distribution of min(running_sd / sigma_f):')
q = res['min_sd_over_sigma_f'].quantile([0, .01, .05, .25, .50, .75, 1.0])
for k, v in q.items():
    print(f'    p{int(k*100):<3} {v:.4f}')

print(f'\n  how many features ever got within a factor of the floor:')
for thr in (0.05, 0.1, 0.2, 0.5, 1.0):
    n = int((res['min_sd_over_sigma_f'] < thr).sum())
    print(f'    min ratio < {thr:<5} : {n:>5}  ({n/tot:.2%})')

if ever:
    print('\n' + '-' * 96)
    print('FEATURES WHERE THE FLOOR BOUND  (worst 40 by pct of dates)')
    print('-' * 96)
    bad = res[res['n_floor_bound'] > 0].nlargest(40, 'pct_floor_bound')
    print(bad[['table_source', 'feature', 'rung', 'sigma_f', 'n_floor_bound',
               'n_active_dates', 'pct_floor_bound', 'min_sd_over_sigma_f']]
          .to_string(index=False, formatters={'pct_floor_bound': '{:.2%}'.format,
                                              'sigma_f': '{:.4g}'.format,
                                              'min_sd_over_sigma_f': '{:.4f}'.format}))
else:
    print('\n  The floor never bound for any feature on any date.')
    print('  It is inert in this pipeline: removing it would not change one z-score.')

print(f'\n  saved -> {OUT_DIR / "floor_diagnostic.csv"}')

# ── cross-reference against the exclusion decisions, if present ──────────────
dec_p = OUT_DIR / 'decisions_aggregate.csv'
if dec_p.exists() and ever:
    dec = pd.concat([pd.read_csv(OUT_DIR / 'decisions_aggregate.csv'),
                 pd.read_csv(ROOT / 'Stage_4_Normalised_Panel' / 'decisions_panel.csv')],
                ignore_index=True)
    m = res[res['n_floor_bound'] > 0].merge(
        dec[['table_source', 'feature', 'action', 'rule_fired', 'std_of_z_ex_capped']],
        on=['table_source', 'feature'], how='left')
    print('\n' + '-' * 96)
    print('WERE THE FLOOR-BOUND FEATURES ALREADY DROPPED BY THE RULES?')
    print('-' * 96)
    print(m['action'].value_counts().to_string())
    print()
    print(m[['feature', 'pct_floor_bound', 'std_of_z_ex_capped', 'action', 'rule_fired']]
          .to_string(index=False, formatters={'pct_floor_bound': '{:.2%}'.format}))

FLOOR DIAGNOSTIC   denom = max(running_sd, 0.1 * sigma_f, 1e-8)
  n_floor_bound = dates where 0.1*sigma_f was used INSTEAD of the running sd
  min_sd_over_sigma_f = how close the running sd ever got to the floor
  (the floor binds when this ratio drops below 0.1)

  agg_market_daily_means               327 features   floor bound on    0   eps bound on   1   min ratio 0.4763
  agg_market_daily_full_moments        991 features   floor bound on    0   eps bound on   1   min ratio 0.3148
  weekly_raw                            34 features   floor bound on    0   eps bound on   0   min ratio 0.4828
  agg_market_monthly_means             320 features   floor bound on    0   eps bound on   1   min ratio 0.0000
  agg_market_monthly_full_moments     1052 features   floor bound on    0   eps bound on   6   min ratio 0.0000
  panel_stock_daily_engineered         192 features   floor bound on    2   eps bound on   0   min ratio 0.0318
  panel_stock_monthly_engineered       197 features   floor bou